In [9]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.types import Send 
import operator

In [10]:
from typing import Union

class State(TypedDict):
    words: list[str]
    output : Annotated[list[dict[str, Union[str, int]]], operator.add] #이렇게 안하면 마지막 값만 남게됨 


graph_builder = StateGraph(State)

In [11]:
# 노드는 그 자리에서 state를 받을 수 있음. (당연한말)
# 아래 노드 생성(이게 우리 작업 단위임)

def node_one(state: State):
    print(f"I want to count {len(state["words"])} words in my state.")
    return {}

def node_two(word: str):
    return { # 같은 타입을 리턴행줘야함 
        "output" : [ # output 업데이트 
            {
                "word":word,
                "letters":len(word),
            }
        ]
    }

In [ ]:
# 우리는 위에 만든 node를 그래프에게 줄것임(graph_builder)

graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two) 
 
def dispatcher(state : State):
      #send 함수 초기화 및 메시지를 보내고 싶은 node의 이름을 전달 
      # "node_two" 에ㅔㄱ 우리가 세고 싶은 word를 준다. 
      return [Send("node_two", word ) for word in state["words"]]
      # 어떤 node에 메시지를 보낼 지 정하고(node_two) 그걸 메시지 payload에 넣는다(word) 그리고 그 ㅇ뒤에는 파이썬 list에 ㅇ넣는다
      # 그러고 node_two가 state를 업데이트한다 
graph_builder.add_edge(START, "node_one")
graph_builder.add_conditional_edges("node_one", dispatcher, ["node_two"]) # 몇 번을 갈지 몰라서 사용하는 add_conditional_edges
graph_builder.add_edge("node_two", END)
 


In [13]:
graph = graph_builder.compile()

# input 임 
graph.invoke(
    {
        "words" : ["hello", "world", "how", "are", "you","doing"]
    }
)

# output 은 dict 
# [
#     {"word" : "hello", "letters" : 5},
#     {"word" : "world", "letters" : 5},
#     {"word" : "how", "letters" : 3}
# ]

ValueError: Found edge ending at unknown node `<function node_one at 0x0000023669CCCC20>`